# Direct Logit Attribution: Which Layer Made This Prediction?

Large language models like Llama 3.1 8B are built on the **Transformer residual stream** architecture. Each layer adds its contribution (from attention and MLP sub-layers) to a running residual stream. The final prediction is made by projecting this residual stream through an unembedding matrix.

**Key insight:** Because the residual stream is a *sum* of all layer outputs, the final logit for any token is also a *sum* of contributions from each component:

```
final_hidden = embed + sum_i(attn_i + mlp_i)

logit(token) = final_hidden @ W_U[token]
             = embed @ W_U[token] + sum_i(attn_i @ W_U[token] + mlp_i @ W_U[token])
```

This decomposition is called **Direct Logit Attribution (DLA)**. It lets us answer: *which layers promoted or suppressed the model's final prediction?*

Based on: Nanda, "A Comprehensive Mechanistic Interpretability Explainer" (2022).

## 1. Setup and Imports

In [ ]:
%matplotlib inline

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

num_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
print("Model loaded: {} layers, hidden size {}".format(num_layers, hidden_size))

## 2. Hooking into Attention and MLP Outputs

We register forward hooks on every layer's **self-attention output projection** and **MLP output** to capture each component's contribution to the residual stream. We also capture the initial embedding.

In [ ]:
def run_with_hooks(model, tokenizer, prompt):
    """Run a forward pass and capture attention/MLP outputs from every layer."""
    attn_outputs = {}
    mlp_outputs = {}
    embed_output = {}
    hooks = []

    # Hook the embedding layer
    def embed_hook(module, input, output):
        embed_output["embed"] = output.detach().clone()

    hooks.append(model.model.embed_tokens.register_forward_hook(embed_hook))

    # Hook each layer's attention and MLP
    for layer_idx in range(num_layers):
        layer = model.model.layers[layer_idx]

        def make_attn_hook(idx):
            def hook_fn(module, input, output):
                # output is a tuple; first element is the attention output
                out = output[0] if isinstance(output, tuple) else output
                attn_outputs[idx] = out.detach().clone()
            return hook_fn

        def make_mlp_hook(idx):
            def hook_fn(module, input, output):
                mlp_outputs[idx] = output.detach().clone()
            return hook_fn

        hooks.append(layer.self_attn.register_forward_hook(make_attn_hook(layer_idx)))
        hooks.append(layer.mlp.register_forward_hook(make_mlp_hook(layer_idx)))

    # Forward pass
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)

    # Remove hooks
    for h in hooks:
        h.remove()

    logits = outputs.logits
    predicted_token_id = logits[0, -1, :].argmax().item()
    predicted_token = tokenizer.decode([predicted_token_id])

    return {
        "embed": embed_output["embed"],
        "attn": attn_outputs,
        "mlp": mlp_outputs,
        "logits": logits,
        "predicted_token_id": predicted_token_id,
        "predicted_token": predicted_token,
        "input_ids": inputs["input_ids"],
    }

# Test with our prompt
prompt = "The capital of France is"
result = run_with_hooks(model, tokenizer, prompt)
print("Prompt: '{}'".format(prompt))
print("Predicted next token: '{}'".format(result["predicted_token"]))
print("Captured {} attention outputs and {} MLP outputs".format(
    len(result["attn"]), len(result["mlp"])))

## 3. The Residual Stream Decomposition

The final hidden state at the last token position is a sum:

```
final_hidden = embed + attn_0 + mlp_0 + attn_1 + mlp_1 + ... + attn_31 + mlp_31
```

Because the logit for a token is `final_hidden @ W_U[token]`, and the dot product is linear, we can decompose it:

```
logit(token) = embed @ W_U[token]
             + attn_0 @ W_U[token] + mlp_0 @ W_U[token]
             + attn_1 @ W_U[token] + mlp_1 @ W_U[token]
             + ...
```

Each term tells us how much that component **directly** contributed to the predicted token's logit. Positive = promoting, negative = suppressing.

**Note:** We must also account for the final LayerNorm before the unembedding. Since LayerNorm is not purely linear, we apply a simplification: we project the raw component outputs onto the unembedding direction. This gives us the *direct* contribution (ignoring LayerNorm scaling), which is the standard DLA approximation.

In [ ]:
def compute_dla(result, model):
    """Compute Direct Logit Attribution for the predicted token."""
    predicted_id = result["predicted_token_id"]

    # Get the unembedding vector for the predicted token
    # W_U shape: (vocab_size, hidden_size)
    w_u = model.lm_head.weight[predicted_id].detach().float()

    # Get the last token position for all components
    last_pos = -1

    # Embedding contribution
    embed_vec = result["embed"][0, last_pos, :].float()
    embed_contrib = torch.dot(embed_vec, w_u).item()

    # Attention and MLP contributions per layer
    attn_contribs = []
    mlp_contribs = []

    for layer_idx in range(num_layers):
        attn_vec = result["attn"][layer_idx][0, last_pos, :].float()
        mlp_vec = result["mlp"][layer_idx][0, last_pos, :].float()

        attn_contribs.append(torch.dot(attn_vec, w_u).item())
        mlp_contribs.append(torch.dot(mlp_vec, w_u).item())

    # Verify: sum of contributions should approximate the actual logit
    total_dla = embed_contrib + sum(attn_contribs) + sum(mlp_contribs)
    actual_logit = result["logits"][0, -1, predicted_id].float().item()

    print("Predicted token: '{}' (id={})".format(result["predicted_token"], predicted_id))
    print("Sum of DLA contributions: {:.4f}".format(total_dla))
    print("Actual logit (after LayerNorm): {:.4f}".format(actual_logit))
    print("Difference (due to final LayerNorm): {:.4f}".format(actual_logit - total_dla))

    return {
        "embed": embed_contrib,
        "attn": attn_contribs,
        "mlp": mlp_contribs,
        "total_dla": total_dla,
        "actual_logit": actual_logit,
    }

dla = compute_dla(result, model)
print("\nEmbedding contribution: {:.4f}".format(dla["embed"]))
print("Top 5 attention contributions:")
for idx in sorted(range(num_layers), key=lambda i: abs(dla["attn"][i]), reverse=True)[:5]:
    print("  Layer {}: {:.4f}".format(idx, dla["attn"][idx]))
print("Top 5 MLP contributions:")
for idx in sorted(range(num_layers), key=lambda i: abs(dla["mlp"][i]), reverse=True)[:5]:
    print("  Layer {}: {:.4f}".format(idx, dla["mlp"][idx]))

## 4. Visualization 1: DLA Waterfall Chart

Each bar shows a single component's direct contribution to the predicted token's logit. Positive bars (blue) **promote** the prediction; negative bars (red) **suppress** it. Attention and MLP are shown separately for each layer.

In [ ]:
def plot_dla_waterfall(dla, num_layers, predicted_token, prompt, save_path=None):
    """Plot a bar chart of each component's DLA contribution."""
    # Build labels and values
    labels = ["embed"]
    values = [dla["embed"]]

    for i in range(num_layers):
        labels.append("L{} attn".format(i))
        values.append(dla["attn"][i])
        labels.append("L{} mlp".format(i))
        values.append(dla["mlp"][i])

    values = np.array(values)
    colors = ["#2196F3" if v >= 0 else "#F44336" for v in values]

    fig, ax = plt.subplots(figsize=(20, 6))
    x = np.arange(len(labels))
    ax.bar(x, values, color=colors, width=0.8, edgecolor="none")

    # Only label every 4th tick to avoid clutter
    tick_step = 4
    ax.set_xticks(x[::tick_step])
    ax.set_xticklabels(labels[::tick_step], rotation=45, ha="right", fontsize=7)

    ax.set_ylabel("Contribution to logit", fontsize=12)
    ax.set_title("Direct Logit Attribution for '{}'\nPrompt: '{}'".format(
        predicted_token, prompt), fontsize=14)
    ax.axhline(y=0, color="black", linewidth=0.5)

    # Legend
    pos_patch = mpatches.Patch(color="#2196F3", label="Promoting")
    neg_patch = mpatches.Patch(color="#F44336", label="Suppressing")
    ax.legend(handles=[pos_patch, neg_patch], loc="upper right")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved: {}".format(save_path))
    plt.show()

waterfall_path = os.path.join(OUTPUT_DIR, "chart_20_dla_waterfall.png")
plot_dla_waterfall(dla, num_layers, result["predicted_token"], prompt, save_path=waterfall_path)

## 5. Visualization 2: Cumulative Logit Build-Up

This plot shows how the logit accumulates as we move through the layers. Starting from the embedding contribution, each layer's attention and MLP add (or subtract) from the running total.

In [ ]:
def plot_cumulative_dla(dla, num_layers, predicted_token, prompt, save_path=None):
    """Plot cumulative logit contribution layer by layer."""
    # Build cumulative values: embed, then each layer adds attn + mlp
    cum_values = [dla["embed"]]
    layer_labels = ["embed"]

    running = dla["embed"]
    for i in range(num_layers):
        running += dla["attn"][i]
        cum_values.append(running)
        layer_labels.append("L{} +attn".format(i))

        running += dla["mlp"][i]
        cum_values.append(running)
        layer_labels.append("L{} +mlp".format(i))

    fig, ax = plt.subplots(figsize=(16, 6))
    x = np.arange(len(cum_values))
    ax.plot(x, cum_values, color="#1565C0", linewidth=1.5, alpha=0.9)
    ax.fill_between(x, cum_values, alpha=0.15, color="#1565C0")

    # Mark key points
    ax.axhline(y=0, color="gray", linewidth=0.5, linestyle="--")
    ax.axhline(y=cum_values[-1], color="#E65100", linewidth=1, linestyle="--",
               label="Final DLA sum: {:.2f}".format(cum_values[-1]))

    # Label every 8th tick
    tick_step = 8
    ax.set_xticks(x[::tick_step])
    ax.set_xticklabels(layer_labels[::tick_step], rotation=45, ha="right", fontsize=8)

    ax.set_ylabel("Cumulative logit contribution", fontsize=12)
    ax.set_xlabel("Component (sequential)", fontsize=12)
    ax.set_title("Cumulative DLA for '{}'\nPrompt: '{}'".format(
        predicted_token, prompt), fontsize=14)
    ax.legend(fontsize=11)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved: {}".format(save_path))
    plt.show()

cumulative_path = os.path.join(OUTPUT_DIR, "chart_21_dla_cumulative.png")
plot_cumulative_dla(dla, num_layers, result["predicted_token"], prompt, save_path=cumulative_path)

## 6. Visualization 3: DLA Across Multiple Prompts

Are certain layers consistently important, or is it prompt-specific? We run DLA on several factual prompts and compare the per-layer total contribution (attn + mlp) as a heatmap.

In [ ]:
prompts = [
    "The capital of France is",
    "The largest planet in the solar system is",
    "Water freezes at a temperature of",
    "The author of Romeo and Juliet is",
    "The speed of light is approximately",
]

all_dlas = []
all_predicted = []

for p in prompts:
    print("Running: '{}'".format(p))
    res = run_with_hooks(model, tokenizer, p)
    d = compute_dla(res, model)
    all_dlas.append(d)
    all_predicted.append(res["predicted_token"])
    print("  -> predicted: '{}'\n".format(res["predicted_token"]))

In [ ]:
def plot_multi_prompt_dla(all_dlas, prompts, all_predicted, num_layers, save_path=None):
    """Heatmap of per-layer total DLA (attn+mlp) across multiple prompts."""
    # Build matrix: rows = prompts, cols = layers
    # Each cell = attn_contrib + mlp_contrib for that layer
    matrix = np.zeros((len(prompts), num_layers))
    for i, d in enumerate(all_dlas):
        for j in range(num_layers):
            matrix[i, j] = d["attn"][j] + d["mlp"][j]

    fig, ax = plt.subplots(figsize=(18, 5))

    # Use diverging colormap centered at 0
    vmax = np.max(np.abs(matrix))
    im = ax.imshow(matrix, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)

    ax.set_xticks(range(0, num_layers, 2))
    ax.set_xticklabels(["L{}".format(i) for i in range(0, num_layers, 2)], fontsize=8)
    ax.set_xlabel("Layer", fontsize=12)

    # Y-axis: short prompt labels with predicted token
    y_labels = []
    for p, tok in zip(prompts, all_predicted):
        short = p if len(p) <= 30 else p[:27] + "..."
        y_labels.append("{} -> '{}'".format(short, tok))
    ax.set_yticks(range(len(prompts)))
    ax.set_yticklabels(y_labels, fontsize=9)

    ax.set_title("Per-Layer DLA (attn+mlp) Across Prompts\nBlue=promoting, Red=suppressing", fontsize=13)

    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Logit contribution", fontsize=10)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved: {}".format(save_path))
    plt.show()

multi_path = os.path.join(OUTPUT_DIR, "chart_22_dla_multi.png")
plot_multi_prompt_dla(all_dlas, prompts, all_predicted, num_layers, save_path=multi_path)

## Summary

In this notebook we implemented **Direct Logit Attribution (DLA)** for Llama 3.1 8B:

1. **Hooked into every layer** to capture the raw attention and MLP outputs that flow into the residual stream.
2. **Decomposed the final logit** by projecting each component's output onto the unembedding vector of the predicted token. This leverages the linearity of the residual stream: `logit = embed @ W_U + sum(attn_i @ W_U + mlp_i @ W_U)`.
3. **Waterfall chart** revealed which specific components most strongly promote or suppress the predicted token.
4. **Cumulative plot** showed how the logit builds up (and sometimes dips) as we traverse the 32 layers.
5. **Multi-prompt comparison** identified whether certain layers are consistently important across different factual recall tasks, or whether importance is prompt-specific.

**Key takeaways:**
- Not all layers contribute equally. A small number of layers tend to dominate the final prediction.
- The MLP layers often carry the "factual knowledge," while attention layers handle routing and context.
- The difference between the DLA sum and the actual logit reflects the non-linear effect of the final LayerNorm, which rescales the residual stream before unembedding.
- DLA is a powerful first step in mechanistic interpretability -- it tells us *where* to look, which we can then investigate with more targeted techniques like activation patching.